### Load libraries

In [ ]:
import json
import yaml

from mstr_robotics._paths import CONFIG_DIR, USER_CONFIG
from mstr_robotics.mstr_classes import MstrGlobal, get_conn
from mstr_robotics._helper import Misc
from mstr_robotics.redis_db import  RedisBiAnalysis,RedisMstrJson
from mstr_robotics._connectors import MstrApi
from mstr_robotics.prepare_ai_data import ExportMstrMd


i_msic=Misc()
i_mstr_global=MstrGlobal()
i_mstr_api=MstrApi()
i_redis_mstr_json=RedisMstrJson()
i_export_mstr_md=ExportMstrMd()

try:
    with open(USER_CONFIG, 'r') as openfile:
        user_d = yaml.safe_load(openfile)
    conn_params =  user_d["conn_params"]
except Exception as err:
    print(err)

try:
    with open(CONFIG_DIR / "mstr_redis_y.yml", 'r') as openfile:
        mstr_redis_y = yaml.safe_load(openfile)
except Exception as err:
    print(err)

try:
    with open(CONFIG_DIR / "jupyter_objects_d.yml", "r") as openfile:
        nb_d = yaml.safe_load(openfile)
    nb_d = nb_d["jup_prj_obj_exporter"]
except Exception as err:
    print(err)


## Config

In [ ]:
# Specifies the synchronisation 
# between MSTR MD and the Redis Db
load_type_fg='daily_search_update'
load_type_fg='full_load'
load_type_fg='shortcut_folder'
load_type_fg='single_objects'

In [ ]:
# definies which MSTR - project is stored where
# in Redis
redis_con_d=mstr_redis_y["redis_env_d"]["redis_dev"]
project_prefix=mstr_redis_y["project_prefix"]
project_id=redis_con_d["project_id"]
prefix_map=mstr_redis_y["prefix_map"]
searches_used_in_prp_d_l=mstr_redis_y["searches_used_in_prp_d_l"]


### Connect to MSTR & Redis

In [ ]:
conn = get_conn(**conn_params)

i_redis_bi_analysis = RedisBiAnalysis( 
    host=redis_con_d["host"],
    port=redis_con_d["port"],
    password=redis_con_d["password"],
    username=redis_con_d["username"],
    decode_responses=redis_con_d["decode_responses"]
)

### Daily Search Update

In [ ]:


if load_type_fg=="daily_search_update":
    err_d_l=[]
    for pre in project_prefix:
        env_prefix=project_prefix[pre]
        conn.select_project(project_id)
        search_result = i_mstr_api.run_mstr_search(conn=conn,
                            search_id=nb_d["misc"]["daily_search_id"])
    
        if search_result["totalItems"] > 0:
            all_obj_d_l=search_result["result"]
    
            err_d_l.append(i_redis_mstr_json.save_obj_json_to_redis(i_redis_bi_analysis=i_redis_bi_analysis                                                    
                                                            ,conn=conn
                                                            ,prefix_map=prefix_map
                                                            , all_obj_d_l=all_obj_d_l
                                                            , env_prefix=env_prefix)
                            )

### Full load

In [ ]:

#load_type_fg="full_load"
if load_type_fg=="full_load":
    # Initialize Redis connection
    env_prefix="mstr_test"
    i_redis_bi_analysis.emergency_flush_db(confirm_phrase='FLUSH_ALL_DATA')
    
    for pre in project_prefix:
        env_prefix=project_prefix[pre]
        conn.select_project(pre)
        all_obj_d_l=i_export_mstr_md.read_out_prj_by_type(conn)
    
        err_d_l=i_redis_mstr_json.save_obj_json_to_redis(i_redis_bi_analysis=i_redis_bi_analysis
                                                        ,conn=conn
                                                        ,prefix_map=prefix_map
                                                        , all_obj_d_l=all_obj_d_l
                                                        , env_prefix=env_prefix)
    
        i_redis_mstr_json.run_searches_redis(conn=conn
                                            ,i_redis_bi_analysis=i_redis_bi_analysis
                                            ,prefix_map=prefix_map
                                            ,env_prefix=env_prefix
                                            ,searches_used_in_prp_d_l=searches_used_in_prp_d_l)
      

### Shortcut Folder

In [ ]:
if load_type_fg=="shortcut_folder":
    folder_id=nb_d["folders"]["folder_id"]
    conn.select_project(project_id)
    all_obj_d_l=i_mstr_global.get_obj_from_sh_fold(conn,folder_id=folder_id)

### Single Objects

In [ ]:
if load_type_fg=="single_objects":
    err_d_l=[]
    for pre in project_prefix:
        env_prefix=project_prefix[pre]
        conn.select_project(project_id)
        all_obj_d_l=[]
        all_obj_d_l.append(nb_d["misc"]["single_object_d"])
        i_redis_mstr_json.save_obj_json_to_redis(i_redis_bi_analysis=i_redis_bi_analysis                                                    
                                                    ,conn=conn
                                                    ,prefix_map=prefix_map
                                                    , all_obj_d_l=all_obj_d_l
                                                    , env_prefix=env_prefix)